# Issue 21 — Qyroven Development replay

Run independently from a clean Colab GPU runtime. This is a technical smoke, not
Final evaluation or a comparison of research effectiveness. Source training was
performed separately in #19; this notebook reuses that exact identified checkpoint.
The accepted inputs remain immutable. See `docs/data/PREQUENTIAL_RUNNER.md`.

The declared matrix uses target 1001, seed42, budgets7/14/28, 29 forecast days, three
candidates, and a ten-tick interruption/resume for budget7. No score-based selection.


In [ ]:
import hashlib
import os
import pathlib
import subprocess
import sys
import zipfile

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import torch

assert torch.cuda.is_available(), "Select a GPU runtime; do not run long training locally"
assert torch.__version__.split("+")[0] == "2.11.0", "Review runtime compatibility explicitly"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "numpy==2.3.3", "ruff==0.17.0", "gdown"], check=True
)
print(sys.version, torch.__version__, torch.cuda.get_device_name(0))


def sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def extract(archive, destination):
    with zipfile.ZipFile(archive) as z:
        assert all(
            not p.startswith("/") and ".." not in pathlib.PurePosixPath(p).parts
            for p in z.namelist()
        )
        z.extractall(destination)

In [ ]:
import gdown

archive = pathlib.Path("/content/inputs-urbanev-SZH-v1.zip")
if not archive.exists():
    gdown.download(id="17pfYqnnSq0Uao2zDNdb8h7GAYfotu_w4", output=str(archive), quiet=False)
assert sha(archive) == "2b8d1a47edebe066543f8f0aec5897c23945edcef62ba10357b1cb95e5ba27c7"
DATA = pathlib.Path("/content/ev-data/data/inputs/issue18-input-handoff-v1-20261019")
if not DATA.exists():
    extract(archive, "/content/ev-data")
source_archive = pathlib.Path("/content/issue19-verified-run.zip")
if not source_archive.exists():
    gdown.download(id="1nTgjpoDwgY78KirC8T5aCcNhmpHDMBWl", output=str(source_archive), quiet=False)
assert sha(source_archive) == "280a2c9c302313b174754a0749824a0ad4a4538663462f309cec85ff77797f75"
extract(source_archive, "/content/issue19-source")
SOURCE = pathlib.Path(
    "/content/issue19-source/qyroven-i19-20261025-seed42-v1/source-seed42/final.pt"
)
SOURCE_SHA = "28d61828a35be69b04f907df9128e7cf4743868060addd113cc61b60363e2905"
assert sha(SOURCE) == SOURCE_SHA

## Identified code

Use an authenticated clone of the repository, then check out the reviewed commit,
or upload the Git bundle exported from the implementation branch. Never paste a
GitHub token into this notebook. The executed runtime commit below contains #19/#20
dependencies and the new #21 implementation, before report-only commits.


In [ ]:
from google.colab import files

uploaded = files.upload()  # Select issue21-code.bundle generated with git bundle create
BUNDLE = pathlib.Path("/content/issue21-code.bundle")
assert sha(BUNDLE) == "e6a95baa60326035eb3e6cfc2a4c6378f5b35df7208e1c862103671e50da3459"
REPO = pathlib.Path("/content/ev-issue21")
assert not REPO.exists(), "Preserve existing checkout; use a fresh runtime/path"
subprocess.run(
    ["git", "clone", "-b", "feat/21-qyroven-prequential-runner", str(BUNDLE), str(REPO)], check=True
)
assert (
    subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
    == "ac493144ded18e0f8419ad002179588af4302d4b"
)
subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=REPO, check=True
)
subprocess.run([sys.executable, "-m", "ruff", "check", "."], cwd=REPO, check=True)

In [ ]:
OUT = pathlib.Path("/content/qyroven-i21-YOUR-UNIQUE-RUN-ID")
assert not OUT.exists()
subprocess.run(
    [
        sys.executable,
        "-m",
        "tools.issue21_colab_smoke",
        "--inputs-root",
        str(DATA),
        "--source",
        str(SOURCE),
        "--source-sha256",
        SOURCE_SHA,
        "--output",
        str(OUT),
    ],
    cwd=REPO,
    check=True,
)
assert (OUT / "matrix.json").is_file(), "Inspect child logs; an attempted run is not success"
print((OUT / "matrix.json").read_text())

## Persist artifacts

Keep checkpoints, events and all forecasts together. Connect only the intended
Google Drive account; the mount can request broad Drive access. Preserve existing
files and verify hashes. Move the resulting ZIP/summary to the team's agreed folder,
then record their links and SHA-256 in the handoff report. Do not claim persistence
until the copy has been flushed and independently verified.


In [ ]:
import shutil

from google.colab import drive

for label, args in [
    ("unit-tests", [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"]),
    ("pip-freeze", [sys.executable, "-m", "pip", "freeze"]),
]:
    result = subprocess.run(args, cwd=REPO, capture_output=True, text=True)
    assert result.returncode == 0
    (OUT / (label + ".txt")).write_text(result.stdout + result.stderr)
(OUT / "SHA256SUMS.txt").write_text(
    "".join(
        sha(p) + "  " + str(p.relative_to(OUT)) + "\n"
        for p in sorted(OUT.rglob("*"))
        if p.is_file()
    )
)
archive_run = pathlib.Path(str(OUT) + ".zip")
with zipfile.ZipFile(archive_run, "x", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file():
            z.write(p, OUT.name + "/" + str(p.relative_to(OUT)))
print("ZIP", archive_run.stat().st_size, sha(archive_run))

drive.mount("/content/drive")
for original, name in [
    (archive_run, archive_run.name),
    (OUT / "matrix.json", OUT.name + "-matrix.json"),
]:
    dest = pathlib.Path("/content/drive/MyDrive") / name
    assert not dest.exists()
    shutil.copyfile(original, dest)
    assert sha(original) == sha(dest)
    print("DRIVE", name, sha(dest))
drive.flush_and_unmount()